## Visualization and stats

In [14]:
import numpy as np 

features = np.load("../features/bipolaire/rem_only/AE129/AE129_REM_1_features.npz")
features.files



['Delta_mean',
 'Delta_var',
 'Delta_std',
 'Delta_skew',
 'Delta_power',
 'Delta_psd_mean',
 'Delta_wavelet',
 'Delta_pca_var',
 'Delta_ica_infomax_energy',
 'Theta_mean',
 'Theta_var',
 'Theta_std',
 'Theta_skew',
 'Theta_power',
 'Theta_psd_mean',
 'Theta_wavelet',
 'Theta_pca_var',
 'Theta_ica_infomax_energy',
 'Alpha_mean',
 'Alpha_var',
 'Alpha_std',
 'Alpha_skew',
 'Alpha_power',
 'Alpha_psd_mean',
 'Alpha_wavelet',
 'Alpha_pca_var',
 'Alpha_ica_infomax_energy',
 'Beta_mean',
 'Beta_var',
 'Beta_std',
 'Beta_skew',
 'Beta_power',
 'Beta_psd_mean',
 'Beta_wavelet',
 'Beta_pca_var',
 'Beta_ica_infomax_energy',
 'Gamma_mean',
 'Gamma_var',
 'Gamma_std',
 'Gamma_skew',
 'Gamma_power',
 'Gamma_psd_mean',
 'Gamma_wavelet',
 'Gamma_pca_var',
 'Gamma_ica_infomax_energy']

In [15]:
len(features)

45

### Puissance spectrale (non-parallélisé)


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Batch REM extraction (FIF + hypnogram .txt) + bipolar montage + per-channel Morlet TFR with autoscale.
+ sauvegarde d'un FULL "clean" (sans artéfacts) et traitement identique pour N2 et N3.

- Cherche automatiquement les patients dans fif_root: {base}/*.fif ou directement *.fif à la racine
  (ou utilise une liste PATIENTS définie à la main).
- Charge le .fif, lit les annotations .txt via get_rem_annotations(base, annot_root)
  pour REM, et get_stage_annotations(..., stages=("N2","N3")) pour N2/N3. """ # --> (A MODIFIER C'EST LE MEME FICHIER POUR LES 3 !!) 
""""
- Construit un FULL "clean" = concat de tous les intervalles qui ne chevauchent pas des annotations
  contenant "BAD" (insensible à la casse), et le SAUVEGARDE : {base}_FULL_clean_noBAD.fif.
- REM : Extrait/concatène les segments REM (print durées, skip si trop court), applique un montage
  bipolaire (MYMONTAGE_BIP), met l'EEG en µV, calcule TFR (Morlet) par canal avec autoscale robuste,
  formatage de l'axe temps en secondes simples, et titre incluant n_epochs.
- N2/N3 : Même pipeline que REM mais en partant du FULL "clean" (mapping temporel).
- Sauvegardes:
    - out_root/{base}/{base}_FULL_clean_noBAD.fif
    - out_root/{base}/{base}_REM_concat_uV.fif
    - out_root/{base}/{base}_{canal}_tfr_REM.png
    - out_root/{base}/{base}_{canal}_tfr_N2.png
    - out_root/{base}/{base}_{canal}_tfr_N3.png
"""

import os
from pathlib import Path
from collections import Counter
import panda as pd
import numpy as np
import mne

# Matplotlib non-interactif si lancé en script
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.ticker import ScalarFormatter
except Exception:
    import matplotlib.pyplot as plt
    from matplotlib.ticker import ScalarFormatter

# ===================== PARAMÈTRES GLOBAUX =====================
fif_root    = Path("/Volumes/Crucial X6/EEG/preprocessed/bipolaire/full") 
annot_root  = Path("/Volumes/Crucial X6/EEG/raw")                         
out_root    = Path("/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD")

# Laisser à None pour auto-découverte, ou donner une liste:
# - soit ["AE129","BJ138",...]
# - soit [("AE129", "/chemin/vers/fichier.fif"), ...] pour pointer un .fif précis
PATIENTS    = None

# TFR
freq_min    = 1.0
freq_max    = 40.0
n_freqs     = 30
cycles_mult = 0.5
epoch_dur   = 4.0
decim       = 2
cmap        = "jet"
vmin, vmax  = None, None       # None => autoscale via percentiles ci-dessous
auto_pct    = (5, 95)

# I/O options
save_rem_fif   = True
overwrite_figs = True

# Seuils de durée pour garder les segments (REM/N2/N3)
MIN_SEG_S   = epoch_dur          # au moins 1 epoch
MIN_TOTAL_S = 2 * epoch_dur      # durée totale minimale après filtrage

# Montage bipolaire — on conserve la même définition que ton script REM
EEG_BIP  = ["Fp2-C4", "C4-O2", "T4-O2", "Cz-Pz", "Fp1-C3", "C3-O1", "Fp1-T3", "T3-O1"]
EOG_BIP  = ["EOGD-A1", "EOGG-A1"]
KEEP_RAW = ["Menton", "JAMBG", "JAMBD", "RONF", "EMG1", "EMG2", "ECG"]

# Après re-référencement bipolaire, on ne filtre PAS par nom "EEG"
RESTRICT_TO_NAME_WITH_EEG = False
# ===================== /PARAMS =====================


# ---- Import de la fonction d'annotations (fallback si besoin) ----

def _read_hypnogram_any(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() == ".txt":
        df = pd.read_csv(path, sep="\t",
                         names=["start", "time", "stage", "index"],
                         engine="python")
        df = df.dropna(subset=["start", "stage"])
        df["start"] = df["start"].astype(float)
        stage = df["stage"]
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, sep=";", engine="python")
        # Colonnes flexibles
        cols = {c.lower().strip(): c for c in df.columns}
        # start en secondes : priorité à l'epoch (30s), sinon delta sur l'heure absolue
        if "position (epoch)" in cols:
            df["start"] = (df[cols["position (epoch)"]].astype(int) - 1) * 30.0
        elif "epoch" in cols:
            df["start"] = (df[cols["epoch"]].astype(int) - 1) * 30.0
        elif "absolute position (hh:mm:ss.ms)" in cols:
            t0 = pd.to_timedelta(df[cols["absolute position (hh:mm:ss.ms)"]].iloc[0])
            df["start"] = (pd.to_timedelta(df[cols["absolute position (hh:mm:ss.ms)"]]) - t0).dt.total_seconds()
        else:
            raise ValueError("CSV hypnogram: colonne epoch/time manquante.")
        # colonne de stade
        stage_col = (cols.get('default staging set ("stage")')
                     or cols.get("stage") or cols.get("stade"))
        stage = df[stage_col]
    else:
        raise ValueError(f"Extension non gérée: {path.suffix}")

    # Normalisation des stades
    def norm(s: str) -> str:
        s = (str(s) or "").strip().upper()
        map_ = {
            "SP": "REM", "R": "REM", "REM": "REM",
            "V": "W", "WAKE": "W", "W": "W",
            "S2": "N2", "STAGE2": "N2", "NREM2": "N2",
            "S3": "N3", "STAGE3": "N3", "NREM3": "N3",
        }
        return map_.get(s, s)

    df = df.assign(stage=stage.map(norm)).sort_values("start")
    df["duration"] = df["start"].shift(-1) - df["start"]
    df = df.iloc[:-1]  # on retire la dernière ligne (durée inconnue)
    return df[["start", "duration", "stage"]]

def _ensure_get_rem_annotations():
    try:
        from src.annotations import get_rem_annotations
        return get_rem_annotations
    except Exception:
        def get_rem_annotations(base_name, annot_dir):
            patient_code = base_name.split("_")[0]
            pdir = Path(annot_dir) / patient_code
            candidates = list(pdir.glob("*.txt")) + list(pdir.glob("*.csv"))
            for p in candidates:
                try:
                    df = _read_hypnogram_any(p)
                    rem = df[df["stage"] == "REM"]
                    if len(rem) > 0:
                        return mne.Annotations(
                            onset=rem["start"].astype(float).tolist(),
                            duration=rem["duration"].astype(float).tolist(),
                            description=["REM"] * len(rem),
                        )
                except Exception:
                    continue
            return None
        return get_rem_annotations

get_rem_annotations = _ensure_get_rem_annotations()

def _ensure_get_stage_annotations():
    import pandas as pd
    def get_stage_annotations(base_name: str, annot_dir: str, stages=("N2", "N3")):
        want = {s.upper() for s in stages}
        patient_code = base_name.split("_")[0]
        pdir = Path(annot_dir) / patient_code
        for p in list(pdir.glob("*.txt")) + list(pdir.glob("*.csv")):
            try:
                df = _read_hypnogram_any(p)
                keep = df[df["stage"].isin(want)]
                if len(keep) == 0:
                    continue
                return mne.Annotations(
                    onset=keep["start"].astype(float).tolist(),
                    duration=keep["duration"].astype(float).tolist(),
                    description=keep["stage"].tolist(),
                )
            except Exception:
                continue
        return None
    return get_stage_annotations

get_stage_annotations = _ensure_get_stage_annotations()



def _sanitize(name: str) -> str:
    return "".join(c for c in name if c.isalnum() or c in ("_", "-")).replace(" ", "")


def discover_patients(fif_root: Path):
    """Retourne une liste [(base, fif_path), ...] en cherchant des .fif.
    - Priorité: sous-dossiers {base}/*.fif (prend le .fif le plus gros)
    - Fallback: fichiers *.fif directement dans fif_root (base = préfixe avant le premier "_")
    """
    mapping = {}
    # 1) sous-dossiers {base}/*.fif
    for child in sorted(fif_root.iterdir()):
        if not child.is_dir():
            continue
        base = child.name
        fif_candidates = [p for p in child.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
        if not fif_candidates:
            continue
        fif_path = max(fif_candidates, key=lambda p: p.stat().st_size)
        mapping[base] = fif_path

    # 2) fichiers *.fif à la racine
    root_fifs = [p for p in fif_root.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
    for p in root_fifs:
        base = p.stem.split("_")[0]
        cur = mapping.get(base)
        if cur is None or p.stat().st_size > cur.stat().st_size:
            mapping[base] = p

    items = sorted(mapping.items())  # [(base, path), ...]
    return items


# ===================== BIPOLAIRE =====================

def _safe_bipolar(inst: mne.io.BaseRaw, anode: str, cathode: str, new_name: str, base: str) -> bool:
    """Crée un canal bipolaire anode-cathode ssi les deux existent."""
    if anode in inst.ch_names and cathode in inst.ch_names:
        try:
            mne.set_bipolar_reference(
                inst, anode=anode, cathode=cathode, ch_name=new_name,
                drop_refs=False, copy=False, verbose="ERROR"
            )
            return True
        except Exception as e:
            print(f"[{base}] Bipolaire {new_name} échec: {e}")
    else:
        missing = [x for x in (anode, cathode) if x not in inst.ch_names]
        print(f"[{base}] Bipolaire {new_name} ignoré (manque {missing})")
    return False


def _apply_bipolar_montage(inst: mne.io.BaseRaw, base: str) -> None:
    """Applique le montage bipolaire MYMONTAGE_BIP + typage des canaux + réduction au set voulu."""
    pairs = [
        ("EOGD", "A1",  "EOGD-A1"),
        ("EOGG", "A1",  "EOGG-A1"),
        ("Fp2",  "C4",  "Fp2-C4"),
        ("C4",   "O2",  "C4-O2"),
        ("T4",   "O2",  "T4-O2"),
        ("Cz",   "Pz",  "Cz-Pz"),
        ("Fp1",  "C3",  "Fp1-C3"),
        ("C3",   "O1",  "C3-O1"),
        ("Fp1",  "T3",  "Fp1-T3"),
        ("T3",   "O1",  "T3-O1"),
    ]

    created = []
    for a, c, n in pairs:
        if _safe_bipolar(inst, a, c, n, base):
            created.append(n)

    # Typage: EEG pour paires EEG, EOG pour EOG*, EMG/ECG pour les capteurs conservés
    eeg_bip = EEG_BIP
    eog_bip = EOG_BIP

    type_map = {}
    for ch in eeg_bip:
        if ch in inst.ch_names:
            type_map[ch] = "eeg"
    for ch in eog_bip:
        if ch in inst.ch_names:
            type_map[ch] = "eog"
    for ch in ("Menton", "EMG1", "EMG2", "JAMBG", "JAMBD"):
        if ch in inst.ch_names:
            type_map[ch] = "emg"
    if "ECG" in inst.ch_names:
        type_map["ECG"] = "ecg"
    if "RONF" in inst.ch_names:
        type_map["RONF"] = "misc"

    if type_map:
        try:
            inst.set_channel_types(type_map)
        except Exception as e:
            print(f"[{base}] set_channel_types après bipolaire: {e}")

    # Réduire strictement aux canaux voulus (ceux créés + capteurs conservés)
    desired = eog_bip + eeg_bip + KEEP_RAW
    present = [ch for ch in desired if ch in inst.ch_names]
    if not present:
        print(f"[{base}] Aucun canal bipolaire/utile présent après montage → rien à faire")
        return
    inst.pick(present)

    print(f"[{base}] Montage bipolaire créé. Canaux conservés ({len(inst.ch_names)}): {inst.ch_names}")


# ===================== OUTILS FULL-CLEAN (BAD*) =====================

def _merge_intervals(intervals):
    """Fusionne des intervalles (start, end) éventuellement chevauchants."""
    if not intervals:
        return []
    ints = sorted(intervals, key=lambda x: x[0])
    merged = [ints[0]]
    for s, e in ints[1:]:
        s0, e0 = merged[-1]
        if s <= e0:
            merged[-1] = (s0, max(e0, e))
        else:
            merged.append((s, e))
    return merged


def build_full_clean(raw_full: mne.io.BaseRaw, base: str): # vérifier l'origine du raw_full 
    """Construit un enregistrement concaténé sans intervalles dont la description contient 'BAD'."""
    sfreq = float(raw_full.info["sfreq"])
    t_end = raw_full.times[-1]
    eps   = 1.0 / sfreq

    bad_intervals = []
    for onset, dur, desc in zip(raw_full.annotations.onset,
                                raw_full.annotations.duration,
                                raw_full.annotations.description):
        if "BAD_ARTIFACT" in (str(desc) or "").upper() and float(dur) > 0:
            s = float(onset)
            e = min(float(onset) + float(dur), t_end)
            if e > s:
                bad_intervals.append((s, e))
    bad_intervals = _merge_intervals(bad_intervals)

    if not bad_intervals:
        print(f"[{base}] Aucun intervalle BAD* détecté (full conservé tel quel).")
        return raw_full.copy(), [(0.0, t_end, 0.0)]

    good = []
    t0 = 0.0
    for (bs, be) in bad_intervals:
        if bs > t0:
            good.append((t0, bs))
        t0 = max(t0, be)
    if t_end > t0:
        good.append((t0, t_end))
    if not good:
        print(f"[{base}] Attention: tout est BAD* → rien à garder.")
        return None, []

    parts, mapping, t_clean = [], [], 0.0
    for (s, e) in good:
        try:
            seg = raw_full.copy().crop(tmin=s, tmax=e - eps, verbose="ERROR")
            parts.append(seg)
            mapping.append((s, e, t_clean))
            t_clean += (e - s)
        except Exception as ex:
            print(f"[{base}] Crop good {s:.2f}-{e:.2f} échoué: {ex}")
    if not parts:
        return None, []

    raw_clean = mne.concatenate_raws(parts, verbose="ERROR")
    print(f"[{base}] Full nettoyé construit: {len(good)} segments bons, durée={t_clean:.2f}s")
    return raw_clean, mapping


def map_intervals_to_clean(intervals, mapping):
    """Mappe des intervalles (s, e) de la timeline ORIGINE vers la timeline NETTOYÉE."""
    out = []
    for (s, e) in intervals:
        if e <= s:
            continue
        for (gs, ge, c0) in mapping:
            a = max(s, gs)
            b = min(e, ge)
            if b > a:
                out.append((c0 + (a - gs), c0 + (b - gs)))
    return out


# ===================== /BIPOLAIRE & FULL-CLEAN =====================


def _find_fif_for_base(base: str) -> Path | None:
    """Cherche un .fif pour `base` dans fif_root/{base}/*.fif, sinon à la racine par préfixe du nom."""
    # 1) dossier du patient
    child = fif_root / base
    if child.is_dir():
        cand = [p for p in child.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
        if cand:
            return max(cand, key=lambda p: p.stat().st_size)
    # 2) racine, par préfixe stem
    cand = [p for p in fif_root.glob(f"{base}*.fif") if p.is_file() and not p.name.startswith("._")]
    if cand:
        return max(cand, key=lambda p: p.stat().st_size)
    return None


def _format_time_axes(figs):
    """Applique l’affichage 'Time (s)' sans puissances de 10 sur les axes temps."""
    for f in figs:
        for ax in f.axes:
            xlabel = (ax.get_xlabel() or "").lower()
            if "time" in xlabel:
                ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=False))
                ax.ticklabel_format(axis="x", style="plain", useOffset=False)
                ax.set_xlabel("Time (s)")


def _plot_and_save_power(power, ch, base, stage, out_png, vmin_eff, vmax_eff, n_epochs):
    """Plot TFR, formate l’axe temps, ajoute le titre avec n_epochs, et sauvegarde."""
    try:
        fig = power.plot(
            picks=[ch], dB=True, cmap=str(cmap),
            vmin=vmin_eff, vmax=vmax_eff,
            baseline=None, show=False
        )
    except TypeError:
        fig = power.plot(picks=[ch], dB=True, cmap=str(cmap),
                         baseline=None, show=False)
        figs_tmp = fig if isinstance(fig, (list, tuple)) else [fig]
        for f in figs_tmp:
            for ax in f.axes:
                artists = list(ax.images) + [c for c in ax.collections if hasattr(c, "set_clim")]
                for art in artists:
                    try:
                        art.set_clim(vmin_eff, vmax_eff)
                    except Exception:
                        pass

    figs = fig if isinstance(fig, (list, tuple)) else [fig]
    _format_time_axes(figs)

    # Titre avec n_epochs
    try:
        figs[0].suptitle(f"{base} — {ch} — {stage}  (n_epochs={n_epochs})", y=0.98)
    except Exception:
        pass

    # Garde-fou "figure blanche"
    ax0 = figs[0].axes[0] if figs and figs[0].axes else None
    is_blank = (ax0 is None) or (len(ax0.images) == 0 and len(ax0.collections) == 0)
    if is_blank:
        print(f"[{base}:{stage}:{ch}] figure vide -> skip (rien sauvegardé)")
        try:
            for f in figs:
                plt.close(f)
        except Exception:
            pass
        return False

    # Sauvegarde
    if isinstance(fig, (list, tuple)):
        fig = fig[0]
    try:
        fig.savefig(out_png, dpi=200, bbox_inches="tight")
        print(f"[{base}:{stage}:{ch}] [ok] {out_png.name}")
    except Exception as e:
        print(f"[{base}:{stage}:{ch}] Save figure erreur: {e}")
    finally:
        plt.close(fig)
    return True


def process_one_patient(item):
    # item peut être "base" (str) ou (base, fif_path)
    if isinstance(item, tuple):
        base, fif_path = item
        fif_path = Path(fif_path)
    else:
        base = str(item)
        fif_path = _find_fif_for_base(base)

    if fif_path is None or not Path(fif_path).exists():
        print(f"[{base}] FIF introuvable -> skip")
        return

    print(f"\n=== {base} ===")
    try:
        raw_full = mne.io.read_raw_fif(fif_path, preload=True, verbose="ERROR")
    except Exception as e:
        print(f"[{base}] Erreur lecture FIF: {e} -> skip")
        return
    print(raw_full)

    # =================== FULL CLEAN (sans BAD*) + SAUVEGARDE ===================
    out_dir = out_root / base
    out_dir.mkdir(parents=True, exist_ok=True)
    clean_fif_path = out_dir / f"{base}_FULL_clean_noBAD.fif"

    raw_clean, mapping = build_full_clean(raw_full, base)
    if raw_clean is None:
        print(f"[{base}] Rien à garder après retrait des BAD* -> skip")
        return
    try:
        raw_clean.save(clean_fif_path, overwrite=True)
        print(f"[{base}] Sauvé: {clean_fif_path.name}")
    except Exception as e:
        print(f"[{base}] Save FULL_clean échoué: {e}")

    # =================== PIPELINE REM (inchangé) ===================
    # Annotations REM depuis .txt (comme avant)
    rem_annots = get_rem_annotations(base, annot_dir=str(annot_root))
    if rem_annots is None or len(rem_annots) == 0:
        print(f"[{base}] Aucune annotation REM -> skip REM")
    else:
        sfreq = float(raw_full.info["sfreq"])
        t_end = raw_full.times[-1]
        eps   = 1.0 / sfreq

        # Filtrage des segments valides
        valid = []
        for onset, dur, desc in zip(rem_annots.onset, rem_annots.duration, rem_annots.description):
            if str(desc).upper() != "REM":
                continue
            if dur is None or dur <= 0:
                continue
            tmin = max(0.0, float(onset))
            tmax = min(tmin + float(dur), t_end) - eps
            if tmax <= tmin:
                continue
            valid.append((tmin, tmax))

        if not valid:
            print(f"[{base}] Aucun segment REM valide -> skip REM")
        else:
            durations = [tmax - tmin for (tmin, tmax) in valid]
            total_dur = float(np.sum(durations))
            print(f"[{base}] REM: Segments valides={len(valid)} | total={total_dur:.2f}s | "
                  f"min={np.min(durations):.2f}s | median={np.median(durations):.2f}s | max={np.max(durations):.2f}s")

            # Filtrage "trop court" selon MIN_SEG_S / MIN_TOTAL_S
            kept = []
            for (tmin, tmax) in valid:
                dur = tmax - tmin
                if dur < MIN_SEG_S:
                    print(f"[{base}]  - REM skip {tmin:.2f}-{tmax:.2f}s (durée {dur:.2f}s < {MIN_SEG_S:.2f}s)")
                else:
                    kept.append((tmin, tmax))
            if not kept or sum(tmax - tmin for (tmin, tmax) in kept) < MIN_TOTAL_S:
                print(f"[{base}] REM: durée après filtrage insuffisante (< {MIN_TOTAL_S:.2f}s) -> skip REM")
            else:
                # Concaténation REM à partir du full original (inchangé)
                rem_raws = []
                for (tmin, tmax) in kept:
                    try:
                        seg = raw_full.copy().crop(tmin=tmin, tmax=tmax, verbose="ERROR")
                        rem_raws.append(seg)
                    except Exception as e:
                        print(f"[{base}] Crop REM {tmin:.2f}-{tmax:.2f}s échoué: {e}")

                if not rem_raws:
                    print(f"[{base}] REM: Aucun segment ajouté -> skip")
                else:
                    rem_raw = mne.concatenate_raws(rem_raws, verbose="ERROR")
                    print(rem_raw)

                    # Montage bipolaire puis typage (identique)
                    _apply_bipolar_montage(rem_raw, base)

                    # --- Ne garder QUE l'EEG par TYPE (les 8 bipolaires listés) ---
                    try:
                        rem_raw.pick_types(
                            meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False,
                            misc=False, resp=False, seeg=False, ecog=False, fnirs=False
                        )
                        if RESTRICT_TO_NAME_WITH_EEG:
                            eeg_names = [ch for ch in rem_raw.ch_names if "EEG" in ch.upper()]
                            if len(eeg_names) == 0:
                                print(f"[{base}] Aucun canal avec 'EEG' dans le nom; conservez tous les EEG typés.")
                            else:
                                rem_raw.pick(eeg_names)
                        print(f"[{base}] Canaux EEG (REM) retenus ({len(rem_raw.ch_names)}): {rem_raw.ch_names}")
                    except Exception as e:
                        print(f"[{base}] Échec du filtrage EEG-only (REM): {e} -> skip REM")
                    else:
                        # Scaling µV (toujours)
                        try:
                            rem_raw.load_data()
                            eeg_picks = mne.pick_types(rem_raw.info, eeg=True, meg=False, eog=False, ecg=False, emg=False)
                            rem_raw.apply_function(lambda x: x * 1e6, picks=eeg_picks, channel_wise=True)
                            if hasattr(rem_raw, "set_unit"):
                                try:
                                    rem_raw.set_unit("eeg", "uV")
                                except Exception:
                                    pass
                        except Exception as e:
                            print(f"[{base}] Échec scaling µV (REM): {e} -> skip REM")
                        else:
                            # I/O
                            if save_rem_fif:
                                try:
                                    out_fif = out_dir / f"{base}_REM_concat_uV.fif"
                                    rem_raw.save(out_fif, overwrite=True)
                                    print(f"[{base}] Sauvé: {out_fif}")
                                except Exception as e:
                                    print(f"[{base}] Échec save FIF (REM): {e}")

                            # TFR (Morlet) — epochs fixes
                            try:
                                epochs = mne.make_fixed_length_epochs(
                                    rem_raw, duration=float(epoch_dur), overlap=0.0, preload=True, verbose="ERROR"
                                )
                                # Ne garder que l'EEG dans epochs
                                epochs.pick_types(meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False, misc=False)
                            except Exception as e:
                                print(f"[{base}] Échec création/filtrage epochs (REM): {e} -> skip TFR REM")
                            else:
                                freqs = np.linspace(float(freq_min), float(freq_max), int(n_freqs))
                                n_cycles = freqs * float(cycles_mult)
                                ch_names = epochs.ch_names
                                n_epochs = len(epochs)
                                print(f"[{base}] Canaux EEG pour TFR REM ({len(ch_names)}): {ch_names}")

                                for ch in ch_names:
                                    out_png = out_dir / f"{base}_{_sanitize(ch)}_tfr_REM.png"
                                    if out_png.exists() and not overwrite_figs:
                                        print(f"[{base}:REM:{ch}] [skip] {out_png.name} existe déjà.")
                                        continue

                                    try:
                                        power = mne.time_frequency.tfr_morlet(
                                            epochs, freqs=freqs, n_cycles=n_cycles,
                                            use_fft=True, return_itc=False, average=True,
                                            picks=[ch], decim=int(decim), verbose="ERROR"
                                        )
                                    except Exception as e:
                                        print(f"[{base}:REM:{ch}] TFR erreur: {e} -> skip")
                                        continue

                                    # Autoscale robuste en dB
                                    try:
                                        Z = 10.0 * np.log10(np.maximum(power.data[0], np.finfo(float).tiny))
                                        if vmin is None or vmax is None:
                                            lo, hi = np.percentile(Z, list(auto_pct))
                                            vmin_eff, vmax_eff = float(lo), float(hi)
                                        else:
                                            vmin_eff, vmax_eff = float(vmin), float(vmax)
                                    except Exception as e:
                                        print(f"[{base}:REM:{ch}] Autoscale erreur: {e} -> skip")
                                        del power
                                        continue

                                    # Plot/format/titre + save
                                    _plot_and_save_power(power, ch, base, "REM", out_png, vmin_eff, vmax_eff, n_epochs)
                                    del power

    # =================== PIPELINE N2 / N3 (à partir du FULL clean) ===================
    stage_ann = get_stage_annotations(base, annot_dir=str(annot_root), stages=("N2", "N3"))
    if stage_ann is None or len(stage_ann) == 0:
        print(f"[{base}] Aucune annotation N2/N3 -> skip N2/N3")
    else:
        # Intervalles N2/N3 sur la timeline ORIGINE (full)
        sfreq_full = float(raw_full.info["sfreq"])
        t_end_full = raw_full.times[-1]
        eps_full = 1.0 / sfreq_full

        by_stage = {"N2": [], "N3": []}
        for onset, dur, desc in zip(stage_ann.onset, stage_ann.duration, stage_ann.description):
            s = max(0.0, float(onset))
            e = min(s + float(dur), t_end_full) - eps_full
            tag = (str(desc) or "").upper()
            if e > s and tag in by_stage:
                by_stage[tag].append((s, e))

        # Pour chaque stade, on mappe vers la timeline NETTOYÉE et on applique le pipeline identique
        for stage in ("N2", "N3"):
            intervals_orig = by_stage.get(stage, [])
            if not intervals_orig:
                print(f"[{base}:{stage}] Aucun intervalle -> skip")
                continue

            intervals_clean = map_intervals_to_clean(intervals_orig, mapping)
            if not intervals_clean:
                print(f"[{base}:{stage}] Intersections avec 'bons' = 0 -> skip")
                continue

            durs = [e - s for (s, e) in intervals_clean if e > s]
            total = float(np.sum(durs)) if durs else 0.0
            print(f"[{base}:{stage}] Segments valides: {len(durs)} | "
                  f"total={total:.2f}s | min={np.min(durs):.2f}s | "
                  f"median={np.median(durs):.2f}s | max={np.max(durs):.2f}s")

            kept = []
            for (s, e) in intervals_clean:
                dur = e - s
                if dur < MIN_SEG_S:
                    print(f"[{base}:{stage}]  - skip seg {s:.2f}-{e:.2f}s (durée {dur:.2f}s < {MIN_SEG_S:.2f}s)")
                else:
                    kept.append((s, e))
            if not kept or sum(e - s for (s, e) in kept) < MIN_TOTAL_S:
                print(f"[{base}:{stage}] Durée après filtrage insuffisante (< {MIN_TOTAL_S:.2f}s) -> skip")
                continue

            # Extraire depuis le FULL clean (et non le full original)
            sfreq_clean = float(raw_clean.info["sfreq"])
            eps_clean   = 1.0 / sfreq_clean
            parts = []
            for (s, e) in kept:
                try:
                    seg = raw_clean.copy().crop(tmin=s, tmax=e - eps_clean, verbose="ERROR")
                    parts.append(seg)
                except Exception as ex:
                    print(f"[{base}:{stage}] Crop {s:.2f}-{e:.2f} échoué: {ex}")
            if not parts:
                print(f"[{base}:{stage}] Rien à concaténer -> skip")
                continue
            stage_raw = mne.concatenate_raws(parts, verbose="ERROR")

            # Montage bipolaire + EEG only + µV (identique à REM)
            _apply_bipolar_montage(stage_raw, base)
            try:
                stage_raw.pick_types(meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False,
                                     misc=False, resp=False, seeg=False, ecog=False, fnirs=False)
                if RESTRICT_TO_NAME_WITH_EEG:
                    eeg_names = [ch for ch in stage_raw.ch_names if "EEG" in ch.upper()]
                    if len(eeg_names) > 0:
                        stage_raw.pick(eeg_names)
                if len(stage_raw.ch_names) == 0:
                    print(f"[{base}:{stage}] Aucun canal EEG bipolaire -> skip")
                    continue
            except Exception as e:
                print(f"[{base}:{stage}] pick_types EEG échoué: {e} -> skip")
                continue

            try:
                stage_raw.load_data()
                eeg_picks = mne.pick_types(stage_raw.info, eeg=True, meg=False, eog=False, ecg=False, emg=False)
                stage_raw.apply_function(lambda x: x * 1e6, picks=eeg_picks, channel_wise=True)
                if hasattr(stage_raw, "set_unit"):
                    try:
                        stage_raw.set_unit("eeg", "uV")
                    except Exception:
                        pass
            except Exception as e:
                print(f"[{base}:{stage}] Échec scaling µV: {e} -> skip")
                continue

            # Epochs + Morlet + autoscale + plots (mêmes réglages que REM)
            try:
                epochs = mne.make_fixed_length_epochs(
                    stage_raw, duration=float(epoch_dur), overlap=0.0, preload=True, verbose="ERROR"
                )
                epochs.pick_types(meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False, misc=False)
            except Exception as e:
                print(f"[{base}:{stage}] Échec création/filtrage epochs: {e} -> skip TFR")
                continue

            n_epochs = len(epochs)
            freqs = np.linspace(float(freq_min), float(freq_max), int(n_freqs))
            n_cycles = freqs * float(cycles_mult)

            for ch in epochs.ch_names:
                out_png = out_dir / f"{base}_{_sanitize(ch)}_tfr_{stage}.png"
                if out_png.exists() and not overwrite_figs:
                    print(f"[{base}:{stage}:{ch}] [skip] {out_png.name} existe déjà.")
                    continue

                try:
                    power = mne.time_frequency.tfr_morlet(
                        epochs, freqs=freqs, n_cycles=n_cycles,
                        use_fft=True, return_itc=False, average=True,
                        picks=[ch], decim=int(decim), verbose="ERROR"
                    )
                except Exception as e:
                    print(f"[{base}:{stage}:{ch}] TFR erreur: {e} -> skip")
                    continue

                try:
                    Z = 10.0 * np.log10(np.maximum(power.data[0], np.finfo(float).tiny))
                    if vmin is None or vmax is None:
                        lo, hi = np.percentile(Z, list(auto_pct))
                        vmin_eff, vmax_eff = float(lo), float(hi)
                    else:
                        vmin_eff, vmax_eff = float(vmin), float(vmax)
                except Exception as e:
                    print(f"[{base}:{stage}:{ch}] Autoscale erreur: {e} -> skip")
                    del power
                    continue

                _plot_and_save_power(power, ch, base, stage, out_png, vmin_eff, vmax_eff, n_epochs)
                del power

    print(f"[{base}] Terminé.")


# ===================== LANCEMENT BATCH =====================
if PATIENTS is None:
    PATIENTS = discover_patients(fif_root)
    bases_preview = [b for b, _ in PATIENTS]
    print(f"Patients détectés ({len(PATIENTS)}): {bases_preview}")

for item in PATIENTS:
    process_one_patient(item)


### Parallélisé

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Batch REM extraction (FIF + hypnogram .txt) + bipolar montage + per-channel Morlet TFR with autoscale.
+ sauvegarde d'un FULL "clean" (sans artéfacts) et traitement identique pour N2 et N3.

- Cherche automatiquement les patients dans fif_root: {base}/*.fif ou directement *.fif à la racine
  (ou utilise une liste PATIENTS définie à la main).
- Charge le .fif, lit les annotations .txt via get_rem_annotations(base, annot_root)
  pour REM, et get_stage_annotations(..., stages=("N2","N3")) pour N2/N3.  # --> (A MODIFIER C'EST LE MEME FICHIER POUR LES 3 !!)

- Construit un FULL "clean" = concat de tous les intervalles qui ne chevauchent pas des annotations
  contenant "BAD" (insensible à la casse), et le SAUVEGARDE : {base}_FULL_clean_noBAD.fif.
- REM : Extrait/concatène les segments REM (print durées, skip si trop court), applique un montage
  bipolaire (MYMONTAGE_BIP), met l'EEG en µV, calcule TFR (Morlet) par canal avec autoscale robuste,
  formatage de l'axe temps en secondes simples, et titre incluant n_epochs.
- N2/N3 : Même pipeline que REM mais en partant du FULL "clean" (mapping temporel).
- Sauvegardes:
    - out_root/{base}/{base}_FULL_clean_noBAD.fif
    - out_root/{base}/{base}_REM_concat_uV.fif
    - out_root/{base}/{base}_{canal}_tfr_REM.png
    - out_root/{base}/{base}_{canal}_tfr_N2.png
    - out_root/{base}/{base}_{canal}_tfr_N3.png
"""

# ==========================
#   Parallélisation & CPU
# ==========================
# À définir AVANT d'importer numpy/scipy/mne
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")  # macOS/Accelerate

import multiprocessing as mp
import faulthandler; faulthandler.enable()  # log des crashes natifs

# ==========================
#   Imports standard
# ==========================
import platform
from pathlib import Path
from collections import Counter
import argparse
import numpy as np
import pandas as pd

# Matplotlib non-interactif sûr en multiprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# Projet / scientifique
import mne
mne.set_config('MNE_MEMMAP_MIN_SIZE', '1M', set_env=True)  # favorise memmap

# ===================== PARAMÈTRES GLOBAUX =====================
# Ces variables seront fixées dynamiquement dans main() après détection du système
fif_root: Path | None = None
annot_root: Path | None = None
out_root: Path | None = None

# Laisser à None pour auto-découverte, ou donner une liste:
# - soit ["AE129","BJ138",...]
# - soit [("AE129", "/chemin/vers/fichier.fif"), ...] pour pointer un .fif précis
PATIENTS = None

# TFR
freq_min    = 1.0
freq_max    = 40.0
n_freqs     = 30
cycles_mult = 0.5
epoch_dur   = 4.0
decim       = 2
cmap        = "jet"
vmin, vmax  = None, None       # None => autoscale via percentiles ci-dessous
auto_pct    = (5, 95)

# I/O options
save_rem_fif   = True
overwrite_figs = True

# Seuils de durée pour garder les segments (REM/N2/N3)
MIN_SEG_S   = epoch_dur          # au moins 1 epoch
MIN_TOTAL_S = 2 * epoch_dur      # durée totale minimale après filtrage

# Montage bipolaire — on conserve la même définition que ton script REM
EEG_BIP  = ["Fp2-C4", "C4-O2", "T4-O2", "Cz-Pz", "Fp1-C3", "C3-O1", "Fp1-T3", "T3-O1"]
EOG_BIP  = ["EOGD-A1", "EOGG-A1"]
KEEP_RAW = ["Menton", "JAMBG", "JAMBD", "RONF", "EMG1", "EMG2", "ECG"]

# Après re-référencement bipolaire, on ne filtre PAS par nom "EEG"
RESTRICT_TO_NAME_WITH_EEG = False
# ===================== /PARAMS =====================


# ---- Import de la fonction d'annotations (fallback si besoin) ----

def _read_hypnogram_any(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() == ".txt":
        df = pd.read_csv(path, sep="\t",
                         names=["start", "time", "stage", "index"],
                         engine="python")
        df = df.dropna(subset=["start", "stage"])
        df["start"] = df["start"].astype(float)
        stage = df["stage"]
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, sep=";", engine="python")
        # Colonnes flexibles
        cols = {c.lower().strip(): c for c in df.columns}
        # start en secondes : priorité à l'epoch (30s), sinon delta sur l'heure absolue
        if "position (epoch)" in cols:
            df["start"] = (df[cols["position (epoch)"]].astype(int) - 1) * 30.0
        elif "epoch" in cols:
            df["start"] = (df[cols["epoch"]].astype(int) - 1) * 30.0
        elif "absolute position (hh:mm:ss.ms)" in cols:
            t0 = pd.to_timedelta(df[cols["absolute position (hh:mm:ss.ms)"]].iloc[0])
            df["start"] = (pd.to_timedelta(df[cols["absolute position (hh:mm:ss.ms)"]]) - t0).dt.total_seconds()
        else:
            raise ValueError("CSV hypnogram: colonne epoch/time manquante.")
        # colonne de stade
        stage_col = (cols.get('default staging set ("stage")')
                     or cols.get("stage") or cols.get("stade"))
        stage = df[stage_col]
    else:
        raise ValueError(f"Extension non gérée: {path.suffix}")

    # Normalisation des stades
    def norm(s: str) -> str:
        s = (str(s) or "").strip().upper()
        map_ = {
            "SP": "REM", "R": "REM", "REM": "REM",
            "V": "W", "WAKE": "W", "W": "W",
            "S2": "N2", "STAGE2": "N2", "NREM2": "N2",
            "S3": "N3", "STAGE3": "N3", "NREM3": "N3",
        }
        return map_.get(s, s)

    df = df.assign(stage=stage.map(norm)).sort_values("start")
    df["duration"] = df["start"].shift(-1) - df["start"]
    df = df.iloc[:-1]  # on retire la dernière ligne (durée inconnue)
    return df[["start", "duration", "stage"]]

def _ensure_get_rem_annotations():
    try:
        from src.annotations import get_rem_annotations
        return get_rem_annotations
    except Exception:
        def get_rem_annotations(base_name, annot_dir):
            patient_code = base_name.split("_")[0]
            pdir = Path(annot_dir) / patient_code
            candidates = list(pdir.glob("*.txt")) + list(pdir.glob("*.csv"))
            for p in candidates:
                try:
                    df = _read_hypnogram_any(p)
                    rem = df[df["stage"] == "REM"]
                    if len(rem) > 0:
                        return mne.Annotations(
                            onset=rem["start"].astype(float).tolist(),
                            duration=rem["duration"].astype(float).tolist(),
                            description=["REM"] * len(rem),
                        )
                except Exception:
                    continue
            return None
        return get_rem_annotations

get_rem_annotations = _ensure_get_rem_annotations()

def _ensure_get_stage_annotations():
    def get_stage_annotations(base_name: str, annot_dir: str, stages=("N2", "N3")):
        want = {s.upper() for s in stages}
        patient_code = base_name.split("_")[0]
        pdir = Path(annot_dir) / patient_code
        for p in list(pdir.glob("*.txt")) + list(pdir.glob("*.csv")):
            try:
                df = _read_hypnogram_any(p)
                keep = df[df["stage"].isin(want)]
                if len(keep) == 0:
                    continue
                return mne.Annotations(
                    onset=keep["start"].astype(float).tolist(),
                    duration=keep["duration"].astype(float).tolist(),
                    description=keep["stage"].tolist(),
                )
            except Exception:
                continue
        return None
    return get_stage_annotations

get_stage_annotations = _ensure_get_stage_annotations()



def _sanitize(name: str) -> str:
    return "".join(c for c in name if c.isalnum() or c in ("_", "-")).replace(" ", "")


def discover_patients(fif_root: Path):
    """Retourne une liste [(base, fif_path), ...] en cherchant des .fif.
    - Priorité: sous-dossiers {base}/*.fif (prend le .fif le plus gros)
    - Fallback: fichiers *.fif directement dans fif_root (base = préfixe avant le premier "_")
    """
    mapping = {}
    # 1) sous-dossiers {base}/*.fif
    for child in sorted(fif_root.iterdir()):
        if not child.is_dir():
            continue
        base = child.name
        fif_candidates = [p for p in child.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
        if not fif_candidates:
            continue
        fif_path = max(fif_candidates, key=lambda p: p.stat().st_size)
        mapping[base] = fif_path

    # 2) fichiers *.fif à la racine
    root_fifs = [p for p in fif_root.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
    for p in root_fifs:
        base = p.stem.split("_")[0]
        cur = mapping.get(base)
        if cur is None or p.stat().st_size > cur.stat().st_size:
            mapping[base] = p

    items = sorted(mapping.items())  # [(base, path), ...]
    return items


# ===================== BIPOLAIRE =====================

def _safe_bipolar(inst: mne.io.BaseRaw, anode: str, cathode: str, new_name: str, base: str) -> bool:
    """Crée un canal bipolaire anode-cathode ssi les deux existent."""
    if anode in inst.ch_names and cathode in inst.ch_names:
        try:
            mne.set_bipolar_reference(
                inst, anode=anode, cathode=cathode, ch_name=new_name,
                drop_refs=False, copy=False, verbose="ERROR"
            )
            return True
        except Exception as e:
            print(f"[{base}] Bipolaire {new_name} échec: {e}")
    else:
        missing = [x for x in (anode, cathode) if x not in inst.ch_names]
        print(f"[{base}] Bipolaire {new_name} ignoré (manque {missing})")
    return False


def _apply_bipolar_montage(inst: mne.io.BaseRaw, base: str) -> None:
    """Applique le montage bipolaire MYMONTAGE_BIP + typage des canaux + réduction au set voulu."""
    pairs = [
        ("EOGD", "A1",  "EOGD-A1"),
        ("EOGG", "A1",  "EOGG-A1"),
        ("Fp2",  "C4",  "Fp2-C4"),
        ("C4",   "O2",  "C4-O2"),
        ("T4",   "O2",  "T4-O2"),
        ("Cz",   "Pz",  "Cz-Pz"),
        ("Fp1",  "C3",  "Fp1-C3"),
        ("C3",   "O1",  "C3-O1"),
        ("Fp1",  "T3",  "Fp1-T3"),
        ("T3",   "O1",  "T3-O1"),
    ]

    created = []
    for a, c, n in pairs:
        if _safe_bipolar(inst, a, c, n, base):
            created.append(n)

    # Typage: EEG pour paires EEG, EOG pour EOG*, EMG/ECG pour les capteurs conservés
    eeg_bip = EEG_BIP
    eog_bip = EOG_BIP

    type_map = {}
    for ch in eeg_bip:
        if ch in inst.ch_names:
            type_map[ch] = "eeg"
    for ch in eog_bip:
        if ch in inst.ch_names:
            type_map[ch] = "eog"
    for ch in ("Menton", "EMG1", "EMG2", "JAMBG", "JAMBD"):
        if ch in inst.ch_names:
            type_map[ch] = "emg"
    if "ECG" in inst.ch_names:
        type_map["ECG"] = "ecg"
    if "RONF" in inst.ch_names:
        type_map["RONF"] = "misc"

    if type_map:
        try:
            inst.set_channel_types(type_map)
        except Exception as e:
            print(f"[{base}] set_channel_types après bipolaire: {e}")

    # Réduire strictement aux canaux voulus (ceux créés + capteurs conservés)
    desired = eog_bip + eeg_bip + KEEP_RAW
    present = [ch for ch in desired if ch in inst.ch_names]
    if not present:
        print(f"[{base}] Aucun canal bipolaire/utile présent après montage → rien à faire")
        return
    inst.pick(present)

    print(f"[{base}] Montage bipolaire créé. Canaux conservés ({len(inst.ch_names)}): {inst.ch_names}")


# ===================== OUTILS FULL-CLEAN (BAD*) =====================

def _merge_intervals(intervals):
    """Fusionne des intervalles (start, end) éventuellement chevauchants."""
    if not intervals:
        return []
    ints = sorted(intervals, key=lambda x: x[0])
    merged = [ints[0]]
    for s, e in ints[1:]:
        s0, e0 = merged[-1]
        if s <= e0:
            merged[-1] = (s0, max(e0, e))
        else:
            merged.append((s, e))
    return merged


def build_full_clean(raw_full: mne.io.BaseRaw, base: str):  # vérifier l'origine du raw_full
    """Construit un enregistrement concaténé sans intervalles dont la description contient 'BAD'."""
    sfreq = float(raw_full.info["sfreq"])
    t_end = raw_full.times[-1]
    eps   = 1.0 / sfreq

    bad_intervals = []
    for onset, dur, desc in zip(raw_full.annotations.onset,
                                raw_full.annotations.duration,
                                raw_full.annotations.description):
        if "BAD_ARTIFACT" in (str(desc) or "").upper() and float(dur) > 0:
            s = float(onset)
            e = min(float(onset) + float(dur), t_end)
            if e > s:
                bad_intervals.append((s, e))
    bad_intervals = _merge_intervals(bad_intervals)

    if not bad_intervals:
        print(f"[{base}] Aucun intervalle BAD* détecté (full conservé tel quel).")
        return raw_full.copy(), [(0.0, t_end, 0.0)]

    good = []
    t0 = 0.0
    for (bs, be) in bad_intervals:
        if bs > t0:
            good.append((t0, bs))
        t0 = max(t0, be)
    if t_end > t0:
        good.append((t0, t_end))
    if not good:
        print(f"[{base}] Attention: tout est BAD* → rien à garder.")
        return None, []

    parts, mapping, t_clean = [], [], 0.0
    for (s, e) in good:
        try:
            seg = raw_full.copy().crop(tmin=s, tmax=e - eps, verbose="ERROR")
            parts.append(seg)
            mapping.append((s, e, t_clean))
            t_clean += (e - s)
        except Exception as ex:
            print(f"[{base}] Crop good {s:.2f}-{e:.2f} échoué: {ex}")
    if not parts:
        return None, []

    raw_clean = mne.concatenate_raws(parts, verbose="ERROR")
    print(f"[{base}] Full nettoyé construit: {len(good)} segments bons, durée={t_clean:.2f}s")
    return raw_clean, mapping


def map_intervals_to_clean(intervals, mapping):
    """Mappe des intervalles (s, e) de la timeline ORIGINE vers la timeline NETTOYÉE."""
    out = []
    for (s, e) in intervals:
        if e <= s:
            continue
        for (gs, ge, c0) in mapping:
            a = max(s, gs)
            b = min(e, ge)
            if b > a:
                out.append((c0 + (a - gs), c0 + (b - gs)))
    return out


# ===================== /BIPOLAIRE & FULL-CLEAN =====================


def _find_fif_for_base(base: str) -> Path | None:
    """Cherche un .fif pour `base` dans fif_root/{base}/*.fif, sinon à la racine par préfixe du nom."""
    global fif_root
    assert fif_root is not None
    # 1) dossier du patient
    child = fif_root / base
    if child.is_dir():
        cand = [p for p in child.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
        if cand:
            return max(cand, key=lambda p: p.stat().st_size)
    # 2) racine, par préfixe stem
    cand = [p for p in fif_root.glob(f"{base}*.fif") if p.is_file() and not p.name.startswith("._")]
    if cand:
        return max(cand, key=lambda p: p.stat().st_size)
    return None


def _format_time_axes(figs):
    """Applique l’affichage 'Time (s)' sans puissances de 10 sur les axes temps."""
    for f in figs:
        for ax in f.axes:
            xlabel = (ax.get_xlabel() or "").lower()
            if "time" in xlabel:
                ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=False))
                ax.ticklabel_format(axis="x", style="plain", useOffset=False)
                ax.set_xlabel("Time (s)")


def _plot_and_save_power(power, ch, base, stage, out_png, vmin_eff, vmax_eff, n_epochs):
    """Plot TFR, formate l’axe temps, ajoute le titre avec n_epochs, et sauvegarde."""
    try:
        fig = power.plot(
            picks=[ch], dB=True, cmap=str(cmap),
            vmin=vmin_eff, vmax=vmax_eff,
            baseline=None, show=False
        )
    except TypeError:
        fig = power.plot(picks=[ch], dB=True, cmap=str(cmap),
                         baseline=None, show=False)
        figs_tmp = fig if isinstance(fig, (list, tuple)) else [fig]
        for f in figs_tmp:
            for ax in f.axes:
                artists = list(ax.images) + [c for c in ax.collections if hasattr(c, "set_clim")]
                for art in artists:
                    try:
                        art.set_clim(vmin_eff, vmax_eff)
                    except Exception:
                        pass

    figs = fig if isinstance(fig, (list, tuple)) else [fig]
    _format_time_axes(figs)

    # Titre avec n_epochs
    try:
        figs[0].suptitle(f"{base} — {ch} — {stage}  (n_epochs={n_epochs})", y=0.98)
    except Exception:
        pass

    # Garde-fou "figure blanche"
    ax0 = figs[0].axes[0] if figs and figs[0].axes else None
    is_blank = (ax0 is None) or (len(ax0.images) == 0 and len(ax0.collections) == 0)
    if is_blank:
        print(f"[{base}:{stage}:{ch}] figure vide -> skip (rien sauvegardé)")
        try:
            for f in figs:
                plt.close(f)
        except Exception:
            pass
        return False

    # Sauvegarde
    if isinstance(fig, (list, tuple)):
        fig = fig[0]
    try:
        fig.savefig(out_png, dpi=200, bbox_inches="tight")
        print(f"[{base}:{stage}:{ch}] [ok] {out_png.name}")
    except Exception as e:
        print(f"[{base}:{stage}:{ch}] Save figure erreur: {e}")
    finally:
        plt.close(fig)
    return True


def process_one_patient(item):
    global out_root, annot_root
    # item peut être "base" (str) ou (base, fif_path)
    if isinstance(item, tuple):
        base, fif_path = item
        fif_path = Path(fif_path)
    else:
        base = str(item)
        fif_path = _find_fif_for_base(base)

    if fif_path is None or not Path(fif_path).exists():
        print(f"[{base}] FIF introuvable -> skip")
        return

    print(f"\n=== {base} ===")
    try:
        raw_full = mne.io.read_raw_fif(fif_path, preload=True, verbose="ERROR")
    except Exception as e:
        print(f"[{base}] Erreur lecture FIF: {e} -> skip")
        return
    print(raw_full)

    # =================== FULL CLEAN (sans BAD*) + SAUVEGARDE ===================
    out_dir = out_root / base
    out_dir.mkdir(parents=True, exist_ok=True)
    clean_fif_path = out_dir / f"{base}_FULL_clean_noBAD.fif"

    raw_clean, mapping = build_full_clean(raw_full, base)
    if raw_clean is None:
        print(f"[{base}] Rien à garder après retrait des BAD* -> skip")
        return
    try:
        raw_clean.save(clean_fif_path, overwrite=True)
        print(f"[{base}] Sauvé: {clean_fif_path.name}")
    except Exception as e:
        print(f"[{base}] Save FULL_clean échoué: {e}")

    # =================== PIPELINE REM (inchangé) ===================
    # Annotations REM depuis .txt (comme avant)
    rem_annots = get_rem_annotations(base, annot_dir=str(annot_root))
    if rem_annots is None or len(rem_annots) == 0:
        print(f"[{base}] Aucune annotation REM -> skip REM")
    else:
        sfreq = float(raw_full.info["sfreq"])
        t_end = raw_full.times[-1]
        eps   = 1.0 / sfreq

        # Filtrage des segments valides
        valid = []
        for onset, dur, desc in zip(rem_annots.onset, rem_annots.duration, rem_annots.description):
            if str(desc).upper() != "REM":
                continue
            if dur is None or dur <= 0:
                continue
            tmin = max(0.0, float(onset))
            tmax = min(tmin + float(dur), t_end) - eps
            if tmax <= tmin:
                continue
            valid.append((tmin, tmax))

        if not valid:
            print(f"[{base}] Aucun segment REM valide -> skip REM")
        else:
            durations = [tmax - tmin for (tmin, tmax) in valid]
            total_dur = float(np.sum(durations))
            print(f"[{base}] REM: Segments valides={len(valid)} | total={total_dur:.2f}s | "
                  f"min={np.min(durations):.2f}s | median={np.median(durations):.2f}s | max={np.max(durations):.2f}s")

            # Filtrage "trop court" selon MIN_SEG_S / MIN_TOTAL_S
            kept = []
            for (tmin, tmax) in valid:
                dur = tmax - tmin
                if dur < MIN_SEG_S:
                    print(f"[{base}]  - REM skip {tmin:.2f}-{tmax:.2f}s (durée {dur:.2f}s < {MIN_SEG_S:.2f}s)")
                else:
                    kept.append((tmin, tmax))
            if not kept or sum(tmax - tmin for (tmin, tmax) in kept) < MIN_TOTAL_S:
                print(f"[{base}] REM: durée après filtrage insuffisante (< {MIN_TOTAL_S:.2f}s) -> skip REM")
            else:
                # Concaténation REM à partir du full original (inchangé)
                rem_raws = []
                for (tmin, tmax) in kept:
                    try:
                        seg = raw_full.copy().crop(tmin=tmin, tmax=tmax, verbose="ERROR")
                        rem_raws.append(seg)
                    except Exception as e:
                        print(f"[{base}] Crop REM {tmin:.2f}-{tmax:.2f}s échoué: {e}")

                if not rem_raws:
                    print(f"[{base}] REM: Aucun segment ajouté -> skip")
                else:
                    rem_raw = mne.concatenate_raws(rem_raws, verbose="ERROR")
                    print(rem_raw)

                    # Montage bipolaire puis typage (identique)
                    _apply_bipolar_montage(rem_raw, base)

                    # --- Ne garder QUE l'EEG par TYPE (les 8 bipolaires listés) ---
                    try:
                        rem_raw.pick_types(
                            meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False,
                            misc=False, resp=False, seeg=False, ecog=False, fnirs=False
                        )
                        if RESTRICT_TO_NAME_WITH_EEG:
                            eeg_names = [ch for ch in rem_raw.ch_names if "EEG" in ch.upper()]
                            if len(eeg_names) == 0:
                                print(f"[{base}] Aucun canal avec 'EEG' dans le nom; conservez tous les EEG typés.")
                            else:
                                rem_raw.pick(eeg_names)
                        print(f"[{base}] Canaux EEG (REM) retenus ({len(rem_raw.ch_names)}): {rem_raw.ch_names}")
                    except Exception as e:
                        print(f"[{base}] Échec du filtrage EEG-only (REM): {e} -> skip REM")
                    else:
                        # Scaling µV (toujours)
                        try:
                            rem_raw.load_data()
                            eeg_picks = mne.pick_types(rem_raw.info, eeg=True, meg=False, eog=False, ecg=False, emg=False)
                            rem_raw.apply_function(lambda x: x * 1e6, picks=eeg_picks, channel_wise=True)
                            if hasattr(rem_raw, "set_unit"):
                                try:
                                    rem_raw.set_unit("eeg", "uV")
                                except Exception:
                                    pass
                        except Exception as e:
                            print(f"[{base}] Échec scaling µV (REM): {e} -> skip REM")
                        else:
                            # I/O
                            if save_rem_fif:
                                try:
                                    out_fif = out_dir / f"{base}_REM_concat_uV.fif"
                                    rem_raw.save(out_fif, overwrite=True)
                                    print(f"[{base}] Sauvé: {out_fif}")
                                except Exception as e:
                                    print(f"[{base}] Échec save FIF (REM): {e}")

                            # TFR (Morlet) — epochs fixes
                            try:
                                epochs = mne.make_fixed_length_epochs(
                                    rem_raw, duration=float(epoch_dur), overlap=0.0, preload=True, verbose="ERROR"
                                )
                                # Ne garder que l'EEG dans epochs
                                epochs.pick_types(meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False, misc=False)
                            except Exception as e:
                                print(f"[{base}] Échec création/filtrage epochs (REM): {e} -> skip TFR REM")
                            else:
                                freqs = np.linspace(float(freq_min), float(freq_max), int(n_freqs))
                                n_cycles = freqs * float(cycles_mult)
                                ch_names = epochs.ch_names
                                n_epochs = len(epochs)
                                print(f"[{base}] Canaux EEG pour TFR REM ({len(ch_names)}): {ch_names}")

                                for ch in ch_names:
                                    out_png = out_dir / f"{base}_{_sanitize(ch)}_tfr_REM.png"
                                    if out_png.exists() and not overwrite_figs:
                                        print(f"[{base}:REM:{ch}] [skip] {out_png.name} existe déjà.")
                                        continue

                                    try:
                                        power = mne.time_frequency.tfr_morlet(
                                            epochs, freqs=freqs, n_cycles=n_cycles,
                                            use_fft=True, return_itc=False, average=True,
                                            picks=[ch], decim=int(decim), verbose="ERROR"
                                        )
                                    except Exception as e:
                                        print(f"[{base}:REM:{ch}] TFR erreur: {e} -> skip")
                                        continue

                                    # Autoscale robuste en dB
                                    try:
                                        Z = 10.0 * np.log10(np.maximum(power.data[0], np.finfo(float).tiny))
                                        if vmin is None or vmax is None:
                                            lo, hi = np.percentile(Z, list(auto_pct))
                                            vmin_eff, vmax_eff = float(lo), float(hi)
                                        else:
                                            vmin_eff, vmax_eff = float(vmin), float(vmax)
                                    except Exception as e:
                                        print(f"[{base}:REM:{ch}] Autoscale erreur: {e} -> skip")
                                        del power
                                        continue

                                    # Plot/format/titre + save
                                    _plot_and_save_power(power, ch, base, "REM", out_png, vmin_eff, vmax_eff, n_epochs)
                                    del power

    # =================== PIPELINE N2 / N3 (à partir du FULL clean) ===================
    stage_ann = get_stage_annotations(base, annot_dir=str(annot_root), stages=("N2", "N3"))
    if stage_ann is None or len(stage_ann) == 0:
        print(f"[{base}] Aucune annotation N2/N3 -> skip N2/N3")
    else:
        # Intervalles N2/N3 sur la timeline ORIGINE (full)
        sfreq_full = float(raw_full.info["sfreq"])
        t_end_full = raw_full.times[-1]
        eps_full = 1.0 / sfreq_full

        by_stage = {"N2": [], "N3": []}
        for onset, dur, desc in zip(stage_ann.onset, stage_ann.duration, stage_ann.description):
            s = max(0.0, float(onset))
            e = min(s + float(dur), t_end_full) - eps_full
            tag = (str(desc) or "").upper()
            if e > s and tag in by_stage:
                by_stage[tag].append((s, e))

        # Pour chaque stade, on mappe vers la timeline NETTOYÉE et on applique le pipeline identique
        for stage in ("N2", "N3"):
            intervals_orig = by_stage.get(stage, [])
            if not intervals_orig:
                print(f"[{base}:{stage}] Aucun intervalle -> skip")
                continue

            intervals_clean = map_intervals_to_clean(intervals_orig, mapping)
            if not intervals_clean:
                print(f"[{base}:{stage}] Intersections avec 'bons' = 0 -> skip")
                continue

            durs = [e - s for (s, e) in intervals_clean if e > s]
            total = float(np.sum(durs)) if durs else 0.0
            print(f"[{base}:{stage}] Segments valides: {len(durs)} | "
                  f"total={total:.2f}s | min={np.min(durs):.2f}s | "
                  f"median={np.median(durs):.2f}s | max={np.max(durs):.2f}s")

            kept = []
            for (s, e) in intervals_clean:
                dur = e - s
                if dur < MIN_SEG_S:
                    print(f"[{base}:{stage}]  - skip seg {s:.2f}-{e:.2f}s (durée {dur:.2f}s < {MIN_SEG_S:.2f}s)")
                else:
                    kept.append((s, e))
            if not kept or sum(e - s for (s, e) in kept) < MIN_TOTAL_S:
                print(f"[{base}:{stage}] Durée après filtrage insuffisante (< {MIN_TOTAL_S:.2f}s) -> skip")
                continue

            # Extraire depuis le FULL clean (et non le full original)
            sfreq_clean = float(raw_clean.info["sfreq"])
            eps_clean   = 1.0 / sfreq_clean
            parts = []
            for (s, e) in kept:
                try:
                    seg = raw_clean.copy().crop(tmin=s, tmax=e - eps_clean, verbose="ERROR")
                    parts.append(seg)
                except Exception as ex:
                    print(f"[{base}:{stage}] Crop {s:.2f}-{e:.2f} échoué: {ex}")
            if not parts:
                print(f"[{base}:{stage}] Rien à concaténer -> skip")
                continue
            stage_raw = mne.concatenate_raws(parts, verbose="ERROR")

            # Montage bipolaire + EEG only + µV (identique à REM)
            _apply_bipolar_montage(stage_raw, base)
            try:
                stage_raw.pick_types(meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False,
                                     misc=False, resp=False, seeg=False, ecog=False, fnirs=False)
                if RESTRICT_TO_NAME_WITH_EEG:
                    eeg_names = [ch for ch in stage_raw.ch_names if "EEG" in ch.upper()]
                    if len(eeg_names) > 0:
                        stage_raw.pick(eeg_names)
                if len(stage_raw.ch_names) == 0:
                    print(f"[{base}:{stage}] Aucun canal EEG bipolaire -> skip")
                    continue
            except Exception as e:
                print(f"[{base}:{stage}] pick_types EEG échoué: {e} -> skip")
                continue

            try:
                stage_raw.load_data()
                eeg_picks = mne.pick_types(stage_raw.info, eeg=True, meg=False, eog=False, ecg=False, emg=False)
                stage_raw.apply_function(lambda x: x * 1e6, picks=eeg_picks, channel_wise=True)
                if hasattr(stage_raw, "set_unit"):
                    try:
                        stage_raw.set_unit("eeg", "uV")
                    except Exception:
                        pass
            except Exception as e:
                print(f"[{base}:{stage}] Échec scaling µV: {e} -> skip")
                continue

            # Epochs + Morlet + autoscale + plots (mêmes réglages que REM)
            try:
                epochs = mne.make_fixed_length_epochs(
                    stage_raw, duration=float(epoch_dur), overlap=0.0, preload=True, verbose="ERROR"
                )
                epochs.pick_types(meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False, misc=False)
            except Exception as e:
                print(f"[{base}:{stage}] Échec création/filtrage epochs: {e} -> skip TFR")
                continue

            n_epochs = len(epochs)
            freqs = np.linspace(float(freq_min), float(freq_max), int(n_freqs))
            n_cycles = freqs * float(cycles_mult)

            for ch in epochs.ch_names:
                out_png = out_dir / f"{base}_{_sanitize(ch)}_tfr_{stage}.png"
                if out_png.exists() and not overwrite_figs:
                    print(f"[{base}:{stage}:{ch}] [skip] {out_png.name} existe déjà.")
                    continue

                try:
                    power = mne.time_frequency.tfr_morlet(
                        epochs, freqs=freqs, n_cycles=n_cycles,
                        use_fft=True, return_itc=False, average=True,
                        picks=[ch], decim=int(decim), verbose="ERROR"
                    )
                except Exception as e:
                    print(f"[{base}:{stage}:{ch}] TFR erreur: {e} -> skip")
                    continue

                try:
                    Z = 10.0 * np.log10(np.maximum(power.data[0], np.finfo(float).tiny))
                    if vmin is None or vmax is None:
                        lo, hi = np.percentile(Z, list(auto_pct))
                        vmin_eff, vmax_eff = float(lo), float(hi)
                    else:
                        vmin_eff, vmax_eff = float(vmin), float(vmax)
                except Exception as e:
                    print(f"[{base}:{stage}:{ch}] Autoscale erreur: {e} -> skip")
                    del power
                    continue

                _plot_and_save_power(power, ch, base, stage, out_png, vmin_eff, vmax_eff, n_epochs)
                del power

    print(f"[{base}] Terminé.")


# ===================== DÉTECTION DU SYSTÈME & I/O ROOTS =====================

def detect_disque(explicit: str | None = None) -> str:
    """Détecte le disque/chemin racine selon l'OS, avec possibilité d'override."""
    if explicit:
        return explicit
    system = platform.system()
    if system == "Darwin":
        return "/Volumes/Crucial X6"
    elif system == "Windows":
        return "D:"
    elif system == "Linux":
        # Adapter si besoin; fallback générique
        user = os.getenv("USER") or os.getenv("USERNAME") or ""
        return f"/media/{user}/Crucial X6" if user else "/media/Crucial X6"
    else:
        raise RuntimeError("Système non supporté pour la détection de disque.")


def _warmup_matplotlib():
    """Évite les races sur la création du cache Matplotlib en multiprocess."""
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    matplotlib.get_cachedir()
    fm.findfont('DejaVu Sans', rebuild_if_missing=True)
    fig = plt.figure()
    plt.plot([0, 1], [0, 1])
    import io
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    plt.close(fig)


# ===================== LANCEMENT BATCH (PARALLÈLE) =====================

def _worker_wrapper(item):
    """Wrapper top-level picklable pour spawn: isole le cache MPL par PID et exécute un patient."""
    try:
        mpl_cache = os.path.join(os.getenv("TMPDIR") or "/tmp", f"mplcache_{os.getpid()}")
        os.environ["MPLCONFIGDIR"] = mpl_cache
        os.makedirs(mpl_cache, exist_ok=True)
    except Exception:
        pass
    try:
        return process_one_patient(item)
    except Exception as e:
        import traceback
        raise RuntimeError(f"Worker error on {item}: {e}\n{traceback.format_exc()}") from e


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--workers", type=int, default=4,
                        help="Nb de processus en parallèle (0 => CPU-1)")
    parser.add_argument("--disk", type=str, default=None,
                        help="Chemin racine du disque (override). Si omis: auto-détection par OS.")
    parser.add_argument("--patients", type=str, nargs="*", default=None,
                        help="Liste de bases patients à traiter (sinon auto-discovery).")
    args = parser.parse_args()

    # Détection de disque selon OS (override CLI possible)
    disque = detect_disque(args.disk)

    # Fixe les roots selon le disque détecté
    fif_root    = Path(f"{disque}/EEG/preprocessed/bipolaire/full")
    annot_root  = Path(f"{disque}/EEG/raw")
    out_root    = Path(f"{disque}/EEG/preprocessed/bipolaire/PWD")

    # Vérif d'existence + création
    for p in [fif_root, annot_root]:
        if not p.exists():
            raise SystemExit(f"[CONFIG] Dossier introuvable: {p}")
    out_root.mkdir(parents=True, exist_ok=True)

    # Patients
    if args.patients:
        # Mode liste de bases (strings)
        PATIENTS = [(b, _find_fif_for_base(b)) for b in args.patients]
    if PATIENTS is None:
        PATIENTS = discover_patients(fif_root)
        bases_preview = [b for b, _ in PATIENTS]
        print(f"Patients détectés ({len(PATIENTS)}): {bases_preview}")

    if not PATIENTS:
        raise SystemExit("Aucun patient à traiter.")

    _warmup_matplotlib()

    # Calcul du nombre de workers
    cpu = os.cpu_count() or 1
    max_workers = (cpu - 1) if args.workers in (0, None) else max(1, args.workers)
    max_workers = min(max_workers, len(PATIENTS))
    print(f"[INFO] Lancement en multiprocess avec {max_workers} worker(s) (CPU={cpu}) (maxtasksperchild=1)")

    # Pool spawn, une tâche par patient, recycle les workers
    ctx = mp.get_context("spawn")
    try:
        with ctx.Pool(processes=max_workers, maxtasksperchild=1) as pool:
            for _ in pool.imap_unordered(_worker_wrapper, PATIENTS, chunksize=1):
                pass
    except Exception as e:
        import traceback
        print(f"[POOL ERROR] {e}\n{traceback.format_exc()}")


Attempting to create new mne-python configuration file:
/Users/darryld/.mne/mne-python.json
Could not read the /Users/darryld/.mne/mne-python.json json file during the writing. Assuming it is empty. Got: Expecting value: line 1 column 1 (char 0)


usage: ipykernel_launcher.py [-h] [--workers WORKERS] [--disk DISK]
                             [--patients [PATIENTS ...]]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/darryld/Library/Jupyter/runtime/kernel-v3540ee6daa2f1fac441b51295a8ed85a9b2907b06.json


SystemExit: 2

/Users/darryld/Desktop/Telecom_Paris/Stage_Cerco/Cerco_studies/.venv_cercoEEGs/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [19]:
# === Debug FIF structure (à exécuter dans une nouvelle cellule) ===
from pathlib import Path
import re
import mne

fif_path = Path("/home/darryld/documents/EEG/preprocessed/bipolaire/2_rem_only/BO60/BO60_REM_concat.fif")

try:
    raw = mne.io.read_raw_fif(fif_path, preload=False, verbose="ERROR")
except Exception as e:
    print(f"Erreur lecture FIF: {e}")
    raise

print(raw)
print("sfreq:", raw.info.get("sfreq"), "| nchan:", raw.info.get("nchan"), "| meas_date:", raw.info.get("meas_date"))
print("bads:", raw.info.get("bads"))
try:
    print("montage:", raw.get_montage())
except Exception as e:
    print("montage: (non défini ou erreur)", e)

# Tableau (index, nom, type, unité) des 50 premiers canaux
types = raw.get_channel_types(unique=False)
units = [ch["unit"] for ch in raw.info["chs"]]

# Conversion code unité -> nom lisible
try:
    from mne.io.constants import FIFF
    _unit_map = {getattr(FIFF, k): k for k in dir(FIFF) if k.startswith("FIFF_UNIT_")}
except Exception:
    _unit_map = {}

def unit_name(u):
    return _unit_map.get(u, "NA" if u in (0, None) else str(u))

print("\n--- Channels (first 50) [idx, name, type, unit] ---")
for i, (nm, tp, u) in enumerate(zip(raw.ch_names, types, units)):
    if i >= 50: break
    print(f"{i:>3}  {nm:<20}  {tp:<6}  {unit_name(u)}")

# Ce que voit pick_types() à partir des métadonnées actuelles
picks_eeg = mne.pick_types(raw.info, eeg=True, meg=False, eog=False, ecg=False, emg=False)
print("\nEEG picks by metadata:", picks_eeg, [raw.ch_names[p] for p in picks_eeg] if len(picks_eeg) else [])

# Heuristique: détecter les noms EEG standards (même si le type n'est pas 'eeg')
# On tolère espaces/underscore et préfixe 'EEG '
_std = r"^(A1|A2|M1|M2|Fp1|Fp2|Fz|F1|F2|F3|F4|F5|F6|F7|F8|FCz|FC[1-6]?|Cz|C1|C2|C3|C4|C5|C6|T3|T4|T5|T6|T7|T8|CPz|CP[1-6]?|Pz|P1|P2|P3|P4|P5|P6|P7|P8|POz|PO[1-8]?|Oz|Iz|O1|O2)$"
def norm_name(s: str) -> str:
    return s.replace("EEG ", "").replace("-REF", "").replace(" ", "").replace("_", "")
cand_eeg = [ch for ch in raw.ch_names if re.match(_std, norm_name(ch))]
print("\nEEG candidates by NAME (regex):", cand_eeg)

# Aperçu des annotations embarquées (si présentes)
ann = raw.annotations
if ann is not None and len(ann) > 0:
    from collections import Counter
    cnt = Counter([d.upper() for d in ann.description])
    print("\nAnnotations summary:", cnt)
else:
    print("\nAnnotations: aucune embarquée dans ce FIF")


<Raw | BO60_REM_concat.fif, 17 x 642275 (2508.9 s), ~24 KiB, data not loaded>
sfreq: 256.0 | nchan: 17 | meas_date: 2021-12-09 04:21:48+00:00
bads: []
montage: <DigMontage | 0 extras (headshape), 0 HPIs, 0 fiducials, 8 channels>

--- Channels (first 50) [idx, name, type, unit] ---
  0  EOGD-A1               eog     FIFF_UNIT_V
  1  EOGG-A1               eog     FIFF_UNIT_V
  2  Fp2-C4                eeg     FIFF_UNIT_V
  3  C4-O2                 eeg     FIFF_UNIT_V
  4  T4-O2                 eeg     FIFF_UNIT_V
  5  Cz-Pz                 eeg     FIFF_UNIT_V
  6  Fp1-C3                eeg     FIFF_UNIT_V
  7  C3-O1                 eeg     FIFF_UNIT_V
  8  Fp1-T3                eeg     FIFF_UNIT_V
  9  T3-O1                 eeg     FIFF_UNIT_V
 10  Menton                emg     FIFF_UNIT_V
 11  JAMBG                 emg     FIFF_UNIT_V
 12  JAMBD                 emg     FIFF_UNIT_V
 13  RONF                  misc    FIFF_UNIT_NONE
 14  EMG1                  emg     FIFF_UNIT_V
 15  EMG2 

In [ ]:
# %% ------------------------------------------------------------
# PSD REM par patient depuis fichiers bipolaires déjà prêts (PWD/{base}/*.fif)
# - Si le .fif contient "REM" dans son nom -> on l'utilise tel quel.
# - Sinon -> on extrait les segments REM via annotations .txt, puis PSD.
# -------------------------------------------------------------
from pathlib import Path
import numpy as np
import mne
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, FuncFormatter

# ===================== PARAMS =====================
PWD_ROOT   = Path("/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD")  # input + output
ANNOT_ROOT = Path("/Volumes/Crucial X6/EEG/raw")                          # {base}/*.txt hypnogrammes

# Laisser None pour auto-découverte des patients (sous-dossiers de PWD_ROOT)
PATIENTS   = None  # ex: ["AE129","BJ138"] ou [("AE129","/chemin/vers/fif"), ...]

# Bandes EEG
BANDS = {
    "Delta": (0.5, 4.0),
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta" : (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}

# Welch
psd_fmin  = 0.5
psd_fmax  = 45.0
psd_win_s = 4.0      # fenêtre (s)
psd_ovlp  = 0.5      # recouvrement relatif

# Seuils REM pour extraction si fichier non-REM
MIN_REM_SEG_S   = psd_win_s        # on jette les segments < fenêtre Welch
MIN_REM_TOTAL_S = 2 * psd_win_s    # durée totale minimale après filtrage

# Sauvegarde
overwrite_figs  = False
save_rem_fif    = False            # si True: sauvegarde le REM concat issu du fichier bip
ASSUME_INPUT_IN_UV = True          # True si vos fichiers bip sont déjà en µV
# ===================== /PARAMS =====================


# ---- annotations (fallback simple) ----
def _ensure_get_rem_annotations():
    try:
        from src.annotations import get_rem_annotations
        return get_rem_annotations
    except Exception:
        import pandas as pd
        def load_annotation_file(txt_path: Path):
            df = pd.read_csv(txt_path, sep="\t", names=["start", "temps", "stage", "index"])
            df = df.dropna(subset=["start", "stage"])
            df["duration"] = df["start"].shift(-1) - df["start"]
            df = df[:-1]
            rem_df = df[df["stage"].str.upper().str.strip() == "REM"]
            return rem_df[["start", "duration"]].values

        def get_rem_annotations(base_name, annot_dir):
            patient_code = base_name.split("_")[0]
            txt_dir = Path(annot_dir) / patient_code
            for txt_path in txt_dir.glob("*.txt"):
                try:
                    rem_intervals = load_annotation_file(txt_path)
                    if len(rem_intervals) > 0:
                        return mne.Annotations(
                            onset=[float(s) for s, _ in rem_intervals],
                            duration=[float(d) for _, d in rem_intervals],
                            description=["REM"] * len(rem_intervals)
                        )
                except Exception:
                    continue
            return None
        return get_rem_annotations

get_rem_annotations = _ensure_get_rem_annotations()


# ---- I/O: trouver le .fif bipolaire dans PWD/{base} ----
def _discover_patients_pwd(root: Path):
    """Retourne [(base, fif_path), ...] depuis PWD/{base}."""
    out = []
    for child in sorted(root.iterdir()):
        if not child.is_dir():
            continue
        base = child.name
        # priorité aux fichiers déjà REM + bip + uV
        patterns = [
            "*REM*concat*bip*uV*.fif", "*REM*bip*uV*.fif",
            "*REM*concat*bip*.fif", "*REM*bip*.fif", "*REM*.fif",
            "*bip*uV*.fif", "*bip*.fif", "*.fif"
        ]
        candidates = []
        for pat in patterns:
            candidates = [p for p in child.glob(pat) if p.is_file() and not p.name.startswith("._")]
            if candidates:
                break
        if candidates:
            fif_path = max(candidates, key=lambda p: p.stat().st_size)
            out.append((base, fif_path))
    return out


# ---- PSD (Welch) robuste ----
def compute_psd_db(raw_eeg: mne.io.BaseRaw, fmin: float, fmax: float,
                   win_s: float, ovlp: float):
    sfreq = float(raw_eeg.info["sfreq"])
    n_times = int(raw_eeg.n_times)
    n_win = int(round(win_s * sfreq))
    n_per_seg = int(np.clip(n_win, 2, n_times))

    ovlp = float(np.clip(ovlp, 0.0, 0.99))
    n_overlap = int(np.floor(ovlp * n_per_seg))
    n_overlap = min(max(n_overlap, 0), n_per_seg - 1)

    try:
        spec = raw_eeg.compute_psd(
            method="welch", fmin=fmin, fmax=fmax,
            n_per_seg=n_per_seg, n_overlap=n_overlap,
            picks=None, reject_by_annotation=False, verbose="ERROR",
        )
    except ValueError:
        print("[WARN] Welch a échoué avec overlap; on réessaie sans recouvrement.")
        spec = raw_eeg.compute_psd(
            method="welch", fmin=fmin, fmax=fmax,
            n_per_seg=min(n_per_seg, int(raw_eeg.n_times)), n_overlap=0,
            picks=None, reject_by_annotation=False, verbose="ERROR",
        )
    freqs = spec.freqs
    psd   = spec.get_data()  # µV²/Hz
    psd_db = 10.0 * np.log10(np.maximum(psd, np.finfo(float).tiny))
    return freqs, psd_db


# ---- Figure par bandes, labels X sans 10^x ----
def plot_psd_by_bands(freqs, psd_db, ch_names, bands, out_png, title_prefix=""):
    f_lo, f_hi = float(freqs[0]), float(freqs[-1])
    bands_eff = {k: (max(v[0], f_lo), min(v[1], f_hi))
                 for k, v in bands.items() if min(v[1], f_hi) > max(v[0], f_lo)}
    n_b = len(bands_eff)
    if n_b == 0:
        print("Aucune bande dans la passband -> pas de figure.")
        return False

    n_cols = 2
    n_rows = int(np.ceil(n_b / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 3.2 * n_rows), sharex=False)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n_b:]:
        ax.axis("off")

    for idx, (band_name, (bmin, bmax)) in enumerate(bands_eff.items()):
        ax = axes[idx]
        mask = (freqs >= bmin) & (freqs <= bmax)
        if not np.any(mask):
            ax.set_visible(False)
            continue
        F = freqs[mask]
        M = psd_db[:, mask]  # (n_ch, n_freqs_band)
        mean_curve = M.mean(axis=0)
        p10 = np.percentile(M, 10, axis=0)
        p90 = np.percentile(M, 90, axis=0)

        ax.semilogx(F, mean_curve, lw=1.5)
        ax.fill_between(F, p10, p90, alpha=0.25, linewidth=0)

        # Axe X en log mais labels "normaux"
        ax.xaxis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 3.0, 5.0)))
        ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:g}"))
        ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(1, 10) * 0.1))

        ax.set_title(f"{band_name}  [{bmin:.1f}–{bmax:.1f}] Hz")
        ax.set_xlabel("Fréquence (Hz)")
        ax.set_ylabel("PSD (dB re µV²/Hz)")
        ax.grid(True, which="both", alpha=0.3)

    fig.suptitle(f"{title_prefix}PSD REM — moy ± p10–p90 par bande", y=0.995)
    fig.tight_layout(rect=[0, 0.02, 1, 0.97])
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    return True


# ---- Pipeline patient ----
def process_one_patient(item):
    if isinstance(item, tuple):
        base, fif_path = item
        fif_path = Path(fif_path)
    else:
        base = str(item)
        # si on nous passe juste la base, chercher dans PWD/{base}
        cand = _discover_patients_pwd(PWD_ROOT)
        d = {b: p for b, p in cand}
        fif_path = d.get(base, None)

    if fif_path is None or not fif_path.exists():
        print(f"[{base}] FIF (bip) introuvable dans PWD -> skip")
        return

    out_dir = PWD_ROOT / base
    out_png = out_dir / f"{base}_PSD_bands_REM_bip.png"
    out_fif = out_dir / f"{base}_REM_from_bip_uV.fif"

    # Early skip
    if out_png.exists() and (not save_rem_fif or out_fif.exists()) and not overwrite_figs:
        msg = f"[{base}] Sorties déjà présentes -> skip ({out_png.name}"
        if save_rem_fif:
            msg += f", {out_fif.name}"
        msg += ")"
        print(msg)
        return

    print(f"\n=== {base} ===")
    try:
        raw_full = mne.io.read_raw_fif(fif_path, preload=True, verbose="ERROR")
    except Exception as e:
        print(f"[{base}] Erreur lecture FIF: {e} -> skip")
        return
    print(raw_full)

    # Décider si fichier est déjà REM
    is_rem_already = ("rem" in fif_path.name.lower())
    if is_rem_already:
        print(f"[{base}] Fichier déjà REM → pas d'extraction d'annotations.")
        rem_raw = raw_full
    else:
        # Annotations REM
        rem_annots = get_rem_annotations(base, annot_dir=str(ANNOT_ROOT))
        if rem_annots is None or len(rem_annots) == 0:
            print(f"[{base}] Aucune annotation REM -> skip")
            return

        sfreq = float(raw_full.info["sfreq"])
        t_end = raw_full.times[-1]
        eps   = 1.0 / sfreq

        valid = []
        for onset, dur, desc in zip(rem_annots.onset, rem_annots.duration, rem_annots.description):
            if str(desc).upper() != "REM" or dur is None or dur <= 0:
                continue
            tmin = max(0.0, float(onset))
            tmax = min(tmin + float(dur), t_end) - eps
            if tmax > tmin:
                valid.append((tmin, tmax))

        if not valid:
            print(f"[{base}] Aucun segment REM valide -> skip")
            return

        durations = [tmax - tmin for (tmin, tmax) in valid]
        total_dur = float(np.sum(durations))
        print(f"[{base}] Segments REM valides: {len(valid)} | total={total_dur:.2f}s | "
              f"min={np.min(durations):.2f}s | median={np.median(durations):.2f}s | max={np.max(durations):.2f}s")

        kept = []
        for (tmin, tmax) in valid:
            dur = tmax - tmin
            if dur < MIN_REM_SEG_S:
                print(f"[{base}]  - skip segment {tmin:.2f}-{tmax:.2f}s (durée {dur:.2f}s < {MIN_REM_SEG_S:.2f}s)")
            else:
                kept.append((tmin, tmax))

        if not kept or sum(tmax - tmin for (tmin, tmax) in kept) < MIN_REM_TOTAL_S:
            print(f"[{base}] Durée REM après filtrage insuffisante (< {MIN_REM_TOTAL_S:.2f}s) -> skip")
            return

        # Concaténer
        rem_raws = []
        for (tmin, tmax) in kept:
            try:
                seg = raw_full.copy().crop(tmin=tmin, tmax=tmax, verbose="ERROR")
                rem_raws.append(seg)
            except Exception as e:
                print(f"[{base}] Crop {tmin:.2f}-{tmax:.2f} échoué: {e}")
        if not rem_raws:
            print(f"[{base}] Rien à concaténer -> skip")
            return
        rem_raw = mne.concatenate_raws(rem_raws, verbose="ERROR")

    # Garder seulement les EEG typés (bipolaires)
    try:
        rem_raw.pick_types(meg=False, eeg=True, eog=False, ecg=False, emg=False,
                           stim=False, misc=False, resp=False, seeg=False, ecog=False, fnirs=False)
        if len(rem_raw.ch_names) == 0:
            print(f"[{base}] Aucun canal EEG -> skip")
            return
        print(f"[{base}] Canaux EEG ({len(rem_raw.ch_names)}): {rem_raw.ch_names}")
    except Exception as e:
        print(f"[{base}] pick_types EEG échoué: {e} -> skip")
        return

    # Mise en µV si nécessaire
    if not ASSUME_INPUT_IN_UV:
        try:
            rem_raw.apply_function(lambda x: x * 1e6, picks="eeg", channel_wise=True)  # V -> µV
            if hasattr(rem_raw, "set_unit"):
                try:
                    rem_raw.set_unit("eeg", "uV")
                except Exception:
                    pass
        except Exception as e:
            print(f"[{base}] Échec scaling µV: {e} -> continue quand même")

    # Option: sauvegarder le REM concat issu du bip
    if save_rem_fif and not is_rem_already:
        try:
            rem_raw.save(out_fif, overwrite=True)
            print(f"[{base}] Sauvegardé: {out_fif}")
        except Exception as e:
            print(f"[{base}] Save FIF échoué: {e}")

    # Early skip figure
    if out_png.exists() and not overwrite_figs:
        print(f"[{base}] Figure existe déjà -> skip: {out_png.name}")
        return

    # PSD
    freqs, psd_db = compute_psd_db(rem_raw, psd_fmin, psd_fmax, psd_win_s, psd_ovlp)

    # Figure par bandes
    ok = plot_psd_by_bands(freqs, psd_db, rem_raw.ch_names, BANDS, out_png, title_prefix=f"{base} — ")
    if ok:
        print(f"[{base}] Figure OK: {out_png.name}")
    else:
        print(f"[{base}] Aucune bande plot -> rien sauvegardé")

    print(f"[{base}] Terminé.")


# ===================== LANCEMENT =====================
if PATIENTS is None:
    PATIENTS = _discover_patients_pwd(PWD_ROOT)
    print(f"Patients détectés ({len(PATIENTS)}): {[b for b, _ in PATIENTS]}")
else:
    # Permettre liste de str ou de tuples
    _tmp = []
    for it in PATIENTS:
        if isinstance(it, tuple):
            _tmp.append(it)
        else:
            # chercher le .fif pour cette base
            found = _discover_patients_pwd(PWD_ROOT)
            m = {b: p for b, p in found}
            if it in m:
                _tmp.append((it, m[it]))
            else:
                _tmp.append((it, None))
    PATIENTS = _tmp

for item in PATIENTS:
    process_one_patient(item)


Patients détectés (67): ['AE129', 'AN166', 'BA152', 'BA171', 'BB114', 'BF181', 'BJ138', 'BO60', 'CA169', 'CB165', 'CC175', 'CD164', 'CD28', 'CJP53', 'CM161', 'CP155', 'CS131', 'CS147', 'DA110', 'DA174', 'DI136', 'DJ137', 'DSJ112', 'EF130', 'FP144', 'GD170', 'GH163', 'GR108', 'GS191', 'HG167', 'IS179', 'JB173', 'JLJ177', 'JP141', 'KH113', 'LS162', 'MA27', 'MC154', 'MFJ160', 'MG16', 'MHTK39', 'ML135', 'MM109', 'MN143', 'MP150', 'MRM132', 'MS128', 'NGA157', 'PA139', 'PF133', 'PJC140', 'RB103', 'RD158', 'RG156', 'RH146', 'RJP148', 'SB176', 'SB178', 'SD134', 'SJP172', 'TJ127', 'TLM168', 'TM142', 'TM151', 'TO145', 'VA182', 'VJ149']

=== AE129 ===
<Raw | AE129_REM_concat_bip_uV.fif, 8 x 3002880 (11730.0 s), ~183.3 MiB, data loaded>
[AE129] Fichier déjà REM → pas d'extraction d'annotations.
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[AE129] Canaux EEG (8): ['Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1']
[AE129] Figure OK: AE129_PSD_

## Par canal

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
REM-only FIF -> (1) PSD par canal (sous-figures) + (2) Topomaps de puissance par bande.

- Auto-découvre les patients dans rem_root: rem_only/{base}/{base}_REM_concat*.fif
- Charge le FIF, garde uniquement les EEG, suppose déjà en µV (configurable).
- Welch PSD par canal, figure multi-sous-graphes (une PSD / canal, bandes ombrées).
- Intègre la PSD par bande -> topomaps (auto-montage standard_1020 si possible).
- Sauvegarde 2 PNG par patient dans rem_only/{base}/.

À coller dans Jupyter ou exécuter comme script.
"""

from pathlib import Path
import re
import numpy as np
import mne
import matplotlib.pyplot as plt

# ===================== PARAMÈTRES =====================
rem_root = Path("/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD")

# Si None -> auto-détection de tous les patients ayant *_REM_concat*.fif
PATIENTS = None   # ex: ["AE129","BJ138"]

# Le FIF est-il déjà en µV ? (True si tu as utilisé ton pipeline précédent)
assume_already_uV = True

# Bornes PSD
psd_fmin, psd_fmax = 1.0, 45.0

# Welch
psd_win_s = 4.0          # longueur de fenêtre en secondes
psd_ovlp  = 0.5          # chevauchement (0..1)

# Bandes d'intérêt
BANDS = {
    "Delta": (0.5, 4.0),
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta":  (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}

# Aesthetics
max_cols = 5             # max colonnes pour la grille "PSD par canal"
shade_alpha = 0.08       # opacité des bandes ombrées sur PSD
psd_fig_dpi = 180
topo_fig_dpi = 180
topo_cmap = "viridis"    # colormap pour topomaps
# ===================== /PARAMS =====================


def discover_patients_from_fif(root: Path):
    """Cherche rem_only/{base}/{base}_REM_concat*.fif et retourne la liste des 'base'."""
    bases = []
    for p in sorted(root.glob("*/*_REM_concat*.fif")):
        base = p.parent.name
        if base not in bases:
            bases.append(base)
    return bases


def find_patient_fif(root: Path, base: str) -> Path | None:
    """Retourne le premier .fif correspondant au patient."""
    cands = sorted((root / base).glob(f"{base}_REM_concat*.fif"))
    return cands[0] if cands else None


def normalize_eeg_names(raw: mne.io.BaseRaw):
    """Renomme canaux pour mieux coller au 10-20: 'EEG Fp1'->'Fp1', 'T3'->'T7', etc."""
    mapping = {}
    for ch in raw.ch_names:
        new = ch
        # remove leading "EEG " ou "EEG_"
        new = re.sub(r"^EEG[\s_]+", "", new)
        # espaces -> rien
        new = new.replace(" ", "")
        # anciennes dénominations
        repl = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
        if new in repl:
            new = repl[new]
        # parfois 'Fpz'/'FpZ' etc. -> standardiser la casse: première lettre maj + reste tel quel
        # (on laisse MNE faire le matching le plus souple possible)
        mapping[ch] = new
    raw.rename_channels(mapping)


def set_montage_if_possible(raw: mne.io.BaseRaw):
    """Tente d'appliquer un montage standard_1020 pour permettre les topomaps."""
    try:
        normalize_eeg_names(raw)
        montage = mne.channels.make_standard_montage("standard_1020")
        # sur EDF, il peut rester des canaux non EEG -> on ne garde que l'EEG
        raw.pick("eeg")
        raw.set_montage(montage, on_missing="ignore")
        # check: combien ont des positions
        pos_cnt = sum([ch["loc"] is not None and np.any(ch["loc"][:3]) for ch in raw.info["chs"]])
        if pos_cnt < len(raw.ch_names) // 2:
            print(f"  [montage] Avertissement: peu de positions reconnues ({pos_cnt}/{len(raw.ch_names)})")
        else:
            print(f"  [montage] Positions connues pour {pos_cnt}/{len(raw.ch_names)} canaux.")
    except Exception as e:
        print(f"  [montage] Impossible d'appliquer standard_1020: {e}")


def compute_welch_psd(raw: mne.io.BaseRaw, fmin, fmax, win_s, ovlp):
    sfreq = float(raw.info["sfreq"])
    n_per_seg = max(2, min(int(sfreq * win_s), raw.n_times))
    n_overlap = int(n_per_seg * float(ovlp))
    psd = raw.compute_psd(
        method="welch", fmin=float(fmin), fmax=float(fmax),
        n_per_seg=n_per_seg, n_overlap=n_overlap,
        picks="eeg", verbose="ERROR"
    )
    freqs = psd.freqs
    data = psd.get_data()         # (n_channels, n_freqs), µV^2/Hz si raw en µV
    return freqs, data


def plot_psd_by_channel(base: str, freqs, psd_data, out_dir: Path):
    """Une sous-figure par canal, PSD en dB, bandes ombrées."""
    n_ch, n_f = psd_data.shape
    n_cols = min(max_cols, n_ch)
    n_rows = int(np.ceil(n_ch / n_cols))

    fig_w = 3.2 * n_cols
    fig_h = 2.4 * n_rows
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False)

    # pré-calcul dB
    tiny = np.finfo(float).tiny
    psd_db = 10.0 * np.log10(np.maximum(psd_data, tiny))

    for i in range(n_ch):
        r, c = divmod(i, n_cols)
        ax = axes[r, c]
        ax.semilogx(freqs, psd_db[i], lw=1.0)
        ax.set_xlim(freqs[0], freqs[-1])
        if r == n_rows - 1:
            ax.set_xlabel("Fréquence (Hz)")
        if c == 0:
            ax.set_ylabel("PSD (dB re µV²/Hz)")
        ax.set_title(raw_eeg_names[i], fontsize=9)
        ax.grid(True, alpha=0.25)

        # bandes ombrées
        for (f1, f2) in BANDS.values():
            f1p, f2p = max(f1, freqs[0]), min(f2, freqs[-1])
            if f2p > f1p:
                ax.axvspan(f1p, f2p, color="k", alpha=shade_alpha)

    # cache axes vides
    for j in range(n_ch, n_rows * n_cols):
        r, c = divmod(j, n_cols)
        axes[r, c].axis("off")

    plt.suptitle(f"{base} — PSD par canal (REM)", y=1.02, fontsize=14)
    plt.tight_layout()
    out_png = out_dir / f"{base}_PSD_by_channel_REM.png"
    fig.savefig(out_png, dpi=psd_fig_dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [save] {out_png.name}")


def band_power_from_psd(freqs, psd_data, band):
    """Intègre la PSD sur [fmin, fmax] -> puissance par canal (µV²)."""
    f1, f2 = band
    idx = np.where((freqs >= f1) & (freqs < f2))[0]
    if idx.size == 0:
        return np.zeros(psd_data.shape[0])
    return np.trapz(psd_data[:, idx], freqs[idx], axis=1)


def plot_topomaps(base: str, raw_info, freqs, psd_data, out_dir: Path):
    """Topomap de la puissance par bande (en dB)."""
    # calcule puissance µV² par bande
    band_vals = {}
    for name, (f1, f2) in BANDS.items():
        P = band_power_from_psd(freqs, psd_data, (f1, f2))           # µV²
        P_db = 10.0 * np.log10(np.maximum(P, np.finfo(float).tiny))  # dB re µV²
        band_vals[name] = P_db

    # figure (N bandes)
    n_b = len(BANDS)
    fig, axes = plt.subplots(1, n_b, figsize=(3.2 * n_b, 3.0), squeeze=False)
    axes = axes[0]

    for ax, (name, vals) in zip(axes, band_vals.items()):
        im, cn = mne.viz.plot_topomap(
            vals, raw_info, axes=ax, show=False, cmap=topo_cmap, contours=0, sphere="auto"
        )
        ax.set_title(name)
        mne.viz.utils._add_colorbar(cn, im, ax)

    plt.suptitle(f"{base} — Topomaps de puissance (dB) par bande (REM)", y=1.04, fontsize=14)
    plt.tight_layout()
    out_png = out_dir / f"{base}_Topomaps_band_power_REM.png"
    fig.savefig(out_png, dpi=topo_fig_dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [save] {out_png.name}")


# ===================== BOUCLE PATIENTS =====================
if PATIENTS is None:
    PATIENTS = discover_patients_from_fif(rem_root)
print(f"Patients détectés ({len(PATIENTS)}): {PATIENTS}")

for base in PATIENTS:
    fif_path = find_patient_fif(rem_root, base)
    if fif_path is None:
        print(f"[{base}] pas de *_REM_concat*.fif -> skip")
        continue

    print(f"\n=== {base} ===")
    out_dir = rem_root / base
    out_dir.mkdir(parents=True, exist_ok=True)

    # Charge le FIF
    try:
        raw = mne.io.read_raw_fif(fif_path, preload=False, verbose="ERROR")
    except Exception as e:
        print(f"  [read] erreur: {e} -> skip")
        continue

    # Garde uniquement les EEG
    raw.pick("eeg")
    raw.load_data()

    # (Optionnel) rescale vers µV si nécessaire
    if not assume_already_uV:
        raw.apply_function(lambda x: x * 1e6, picks="eeg", channel_wise=True)

    # Appliquer une montage si possible (pour topomaps)
    set_montage_if_possible(raw)

    # noms des canaux EEG (après normalisation)
    raw_eeg_names = raw.ch_names

    # PSD (Welch)
    freqs, psd = compute_welch_psd(raw, psd_fmin, psd_fmax, psd_win_s, psd_ovlp)

    # Figure 1: PSD par canal (sous-figures)
    plot_psd_by_channel(base, freqs, psd, out_dir)

    # Figure 2: Topomaps de puissance par bande
    try:
        plot_topomaps(base, raw.info, freqs, psd, out_dir)
    except Exception as e:
        print(f"  [topomap] impossible de tracer: {e} (positions manquantes ?)")


### On décortique tout pour que ça se fasse bien